In [9]:
import pandas as pd

def format_number(number, precision=6):
    # Round to the specified precision
    rounded = round(number, precision)
    # Format with the specified precision and remove trailing zeros
    result = f"{rounded:.{precision}f}".rstrip("0").rstrip(".")
    return result


def calculate_reverse_dual_investment(m, n, x, y, premium, High_future_price, Low_future_price):
    """
    Calculate strike, Δx, and Δy for reversed dual investment.

    Parameters:
    - m: Amount of token0 to deposit (e.g., ETH)
    - n: Amount of token1 to deposit (e.g., USDC)
    - x: Initial reserve of token0 (default: 10)
    - y: Initial reserve of token1 (default: 200000)
    - premium: Premium factor (default: 0)

    Returns:
    - strike: Strike price
    - delta_x: Change in token0
    - delta_y: Change in token1
    """
    # Calculate strike
    strike = (y + n) / (x + m) 

    # Calculate Δx and Δy
    delta_x = m * (1 + premium) - n / strike 
    delta_y = n * (1 + premium) - m * strike

    # call (看漲)：
    # exercise: pay m*strike of token1, receive m token0
    exerc_call_swap1in = m * strike
    exerc_call_swap0out = m

    # put（看跌）：
    # exercise: pay n/strike of token0, receive n token1
    exerc_put_swap0in = n / strike
    exerc_put_swap1out = n

    # PnL:
    cost = delta_x * 2000 + delta_y
    
    # price_go_up
    revenue_high = m * (High_future_price - 2000)
    # price_go_down
    revenue_low = (n / 2000) * (2000 - Low_future_price)

    return strike, delta_x, delta_y, exerc_call_swap1in, exerc_call_swap0out, exerc_put_swap0in, exerc_put_swap1out, cost, revenue_high, revenue_low


def main():
    print("====Reversed Dual Investment Calculator with Multiple Values====")

    try:
        m_n_values = [
            (1, 0),   
            (0, 2000),
            (1, 2000),
            (1.1, 2000),
            (1, 2100),
        ]

        # Optional input for initial reserves and premium
        x = 100
        y = 200000
        premium = 0.01

        # Generate all combinations of m and n
        results = []
        for m, n in m_n_values:
            (
                strike,
                delta_x,
                delta_y,
                exerc_call_swap1in,
                exerc_call_swap0out,
                exerc_put_swap0in,
                exerc_put_swap1out,
                cost,
                revenue_high,
                revenue_low
            ) = calculate_reverse_dual_investment(m, n, x, y, premium, 3000, 1000)
            results.append(
                {
                    "m": m,
                    "n": n,
                    "strike": strike,
                    "delta_x": delta_x,
                    "delta_y": delta_y,
                    "行權 call: (swap1in, swap0out)": (
                        format_number(exerc_call_swap1in, precision=4),
                        format_number(exerc_call_swap0out, precision=4),
                    ),
                    "行權 put: (swap0in, swap1out)": (
                        format_number(exerc_put_swap0in, precision=4),
                        format_number(exerc_put_swap1out, precision=4),
                    ),
                    "成本": format_number(cost, precision=4),
                    "若期末價格走揚的 Profit": format_number(revenue_high, precision=4),
                    "若期末價格走低的 Profit": format_number(revenue_low, precision=4)
                }
            )

        # Convert results to DataFrame
        df = pd.DataFrame(results)

        # Display the DataFrame
        print("init status:")
        print("premium:", premium)
        print("init_x:", x)
        print("init_y:", y)
        print("\nResults:")
        return df

    except ValueError as e:
        print("Invalid input! Please enter numeric values.")
    except ZeroDivisionError:
        print("Error: Division by zero! Check input values (x + m cannot be zero).")


df = main()
df

====Reversed Dual Investment Calculator with Multiple Values====
init status:
premium: 0.01
init_x: 100
init_y: 200000

Results:


,m,n,strike,delta_x,delta_y,"行權 call: (swap1in, swap0out)","行權 put: (swap0in, swap1out)",成本,若期末價格走揚的 Profit,若期末價格走低的 Profit
0,1.0,0,1980.198020,1.010000,-1980.198020,"(1980.198, 1)","(0, 0)",39.802,1000,0
1,0.0,2000,2020.000000,-0.990099,2020.000000,"(0, 0)","(0.9901, 2000)",39.802,0,1000
2,1.0,2000,2000.000000,0.010000,20.000000,"(2000, 1)","(1, 2000)",40,1000,1000
3,1.1,2000,1998.021761,0.110010,-177.823937,"(2197.8239, 1.1)","(1.001, 2000)",42.1959,1100,1000
4,1.0,2100,2000.990099,-0.039480,120.009901,"(2000.9901, 1)","(1.0495, 2100)",41.049,1000,1050


- Parameters:
    - m: Target Amount of token0 to deposit (e.g., ETH)
    - n: Target Amount of token1 to deposit (e.g., USDC)
    - x: Initial reserve of token0 (default: 10)
    - y: Initial reserve of token1 (default: 200000)
    - premium: Premium factor (default: 0)
  
反向雙幣deposit時，相當於讓使用者付 premium + 立即行權。期限到期前，在 m, n 皆不為零的情況下，使用者等於是持有雙向的選擇權，手握兩種不同方向的選擇權，所以他可以選擇要對哪邊進行撤銷行權。

### Example1
- m = 1, n = 0
- User Deposit pay = 1 eth
- User Deposit get = 1980.198 USD
- note = (m, n, strike) = 1, 0, 1980.198

使用者理論上有兩個方向的選擇權可以撤銷行權，但因為此例中他只做 put，只能撤銷 put 的行權，另一邊的 call 無法撤銷行權:
- option1: revert exercising Put option = pay 1980.198 U, receive 1 eth
- option2: revert exercising Call option = pay 0 eth, receive 0 U

### Example2
- m = 1, n = 2000
- User Deposit pay = 0.01 eth + 20 U
- User Deposit get = 0
- note = (m, n, strike) = 1, 2000, 2000

使用者有兩個方向的選擇權 可以撤銷行權:
- option1: revert exercising Put option = pay 2000 U, receive 1 eth
- option2: revert exercising Call option = pay 1 eth, receive 2000 U

### Example3
- m = 1, n = 2000
- User Deposit pay = 0.01 eth + 20 U
- User Deposit get = 0
- note = (m, n, strike) = 1, 2000, 2000

使用者有兩個方向的選擇權 可以撤銷行權:
- option1: revert exercising Put option = pay 2000 U, receive 1 eth
- option2: revert exercising Call option = pay 1 eth, receive 2000 U

--- 撤銷行權規則--- 

Call (看漲)：
pay m*strike of token1, receive m token0

Put（看跌）：
pay n/strike of token0, receive n token1
